## Colab session configuration

Connect to google cloud runtime and use GPU.

## Reading hg38 genome assembly 

Import.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys, pathlib, subprocess
from pathlib import Path
!pip install pyfaidx
from pyfaidx import Fasta
from huggingface_hub import hf_hub_download

Configure root.

In [10]:
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/mini-gLM.git"
repo_dir = Path("mini-gLM")

if COLAB:
    root = Path("/content/mini-gLM")

    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])

else:
    root = Path.cwd().parent

sys.path.insert(0, str(root))

Download pre-training data from HF.

In [ ]:
path = hf_hub_download(
    repo_id="eddykang06/hg38-pretraining",
    repo_type="dataset",
    filename="pretraining.csv.gz",
)

pretraining = pd.read_csv(path, compression = "gzip", usecols = ["sequence"])["sequence"].to_list()

c:\Users\eddyk\miniconda3\envs\staix\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\eddyk\.cache\huggingface\hub\datasets--eddykang06--hg38-pretraining. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## Tokenizer

Simple function to generate random samples to mimic fasta samples.

In [29]:
def generate_seqs(min_length, max_length, num_seqs, seed = None):  
    rng = np.random.default_rng(seed)
    seqs = []

    for i in range(num_seqs):
        length = rng.choice(np.arange(min_length, max_length))
        seq = [rng.choice(["A", "C", "G", "T"]) for i in range(length)]
        seq = "".join(seq)
        seqs.append(seq)
        
    return seqs

Train BPE tokenizer.

In [ ]:
from src.tokenize import BPETokenizer


## Model architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Implement simple attention block.

https://medium.com/@heyamit10/implement-self-attention-and-cross-attention-in-pytorch-cfe17ab0b3ee

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        
        # Queries, keys, values
        self.q_map = nn.Linear(d_model, d_model)
        self.k_map = nn.Linear(d_model, d_model)
        self.v_map = nn.Linear(d_model, d_model)

    def forward(self, x):

        # x[B, L, D]          
        # q[B, L, d_model], k[B, L, d_model], v[B, L, d_model]
        q = self.q_map(x)
        k = self.k_map(x)
        v = self.v_map(x)

        # a[B, L, L] (softmax over last dimension L from keys)
        a = torch.softmax(q @ k.transpose(-2, -1) / (self.d_model ** 0.5), dim = -1)

        # y[B, L, d_model]
        out = a @ v

        return out

Implement multi-head attention block.

In [ ]:
class MultiHeadAttention():
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = self.d_model // self.num_heads

        # Queries, keys, values
        self.q_map = nn.Linear(d_model, d_model)
        self.k_map = nn.Linear(d_model, d_model)
        self.v_map = nn.Linear(d_model, d_model)

        # Final FC
        self.o_map = nn.Linear(d_model, d_model)

    def forward(self, x):

        B, L, D = x.shape

        q = self.q_map(x).reshape(B, L, self.num_heads, self.d_head).transpose(1, 2)
        k = self.k_map(x).reshape(B, L, self.num_heads, self.d_head).transpose(1, 2)
        v = self.v_map(x).reshape(B, L, self.num_heads, self.d_head).transpose(1, 2)

        a = torch.softmax(q @ k.transpose(-2, -1) / (self.d_head ** 0.5), dim = -1)
        
        out = a @ v
        out = out.transpose(1, 2).reshape(B, L, D)
        out = self.o_map(out)

        return out

Implement sparse MultiHeadAttention.

In [ ]:
class SparseMultiHeadAttention():
    def __init__(self, d_model, num_heads):
        super().__init__()

Implement transformer encoder block.

In [ ]:
class TransformerEncoder():
    def __init__(self):
        super().__init__()

Implement sparse attention transformer block.

In [ ]:
class SparseTransformerEncoder():
    def __init__(self):
        super().__init__()

Things that will be implemented:
- Padding to fixed max sequence length (512?)
- Embedding module
- Masked token prediction objective
- Sparse attention transformer
- 

Maybe pre-train on approx. 500k-1mil sequences?

Input to model should be a list of token IDs.

In [ ]:
class SparseGLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        
    ):
        super().__init__()

        self.embed = nn.Embedding

        # Transformer layers?
        
    def forward(self, x):

        return


class GLM(nn.Module):
    def __init(
        self,
        vocab_size
    ):
        super().__init__()



## Training

Implement masked language modeling training objective.
- Batching [B, L, D]
    - Token right padding should be done to the max length in the batch* (computational efficient)
- Attention mask to ignore padding tokens
- Train weights in batches

Reference: https://huggingface.co/learn/llm-course/en/chapter6/5